In [ ]:
import json
import pathlib
import re

import pandas as pd

from tqdm.auto import tqdm

from iupac_name_parser import IUPACNameParser

In [ ]:
ROOT_DIR = pathlib.Path.cwd()
RAW_DATA_DIR = ROOT_DIR / "raw_data"
TREATED_DATA_DIR = ROOT_DIR / "treated_data"

ROTATION_RESULTS_DIR = TREATED_DATA_DIR / "rotation_results"
assert ROTATION_RESULTS_DIR.exists()

EXTRACTED_DIR = RAW_DATA_DIR / "extracted"
assert EXTRACTED_DIR.exists()

DATA_BLOCKS_DIR = TREATED_DATA_DIR / "data_blocks"
assert DATA_BLOCKS_DIR.exists()

COLLECTED_BOLD_BLOCKS_PATH = TREATED_DATA_DIR / "collected_bold_blocks.csv"
COLLECTED_ROTATION_RESULTS_PATH = TREATED_DATA_DIR / "collected_rotation_results.csv"

In [ ]:
mol_parser = IUPACNameParser()

In [ ]:
def get_extended_block(
    block: dict, molecule: str, article_id: str, file_stem: str
) -> dict | None:
    try:
        name, smiles, appears_ambiguous, stereo_ignored = mol_parser.to_smiles(molecule)
    except Exception:
        return None
    return {
        "article_id": article_id,
        "file_stem": file_stem,
        **block,
        "molecule": name,
        "smiles": smiles,
        "appears_ambiguous": appears_ambiguous,
        "stereo_ignored": stereo_ignored,
    }

## Collect boldface text blocks

In [ ]:
if COLLECTED_BOLD_BLOCKS_PATH.exists():
    raise FileExistsError(f"File {COLLECTED_BOLD_BLOCKS_PATH} already exists")

files = list(EXTRACTED_DIR.glob("**/*.json"))
articles_with_bold_blocks = []
for file in tqdm(files, desc="Processing bold block files"):
    if file.stat().st_size == 0:
        continue
    with open(file, "r") as f:
        blocks_from_file = json.load(f)
    for block in blocks_from_file:
        extended_block = get_extended_block(
            block, block["text"], file.parent.name, file.stem
        )
        if extended_block is None:
            continue
        articles_with_bold_blocks.append(extended_block)

bold_blocks_df = pd.DataFrame(articles_with_bold_blocks)
bold_blocks_df.to_csv(COLLECTED_BOLD_BLOCKS_PATH, index=False)
bold_blocks_df

## Collect rotation results

In [ ]:
SHORT_THRESHOLD_DISTANCE = 200
LONG_THRESHOLD_DISTANCE = 800

UNITS = r"(?:cm\s*-1|ppm)"
UNITS_PATTERN = re.compile(rf"\(?{UNITS}\)?")

POSITIVE_NUMBER = r"\d+(?:\.\d+)?"
NUMBER_OR_RANGE = rf"{POSITIVE_NUMBER}(?:\s*-\s*{POSITIVE_NUMBER})?"
INSIDE_PARENTHESES = r"\((?:[^()]+)\)"
NMR_PEAK = rf"{NUMBER_OR_RANGE}(?:\s*{INSIDE_PARENTHESES})?"
NMR_PEAK_LIST = rf"{NMR_PEAK}(?:[\s,]+{NMR_PEAK})*"
NMR_PATTERN = re.compile(
    rf"""
        (?:\d*[HCF]\s*)?               # Optional number and H/C/F notation
        NMR                            # NMR literal
        (?:\s*{INSIDE_PARENTHESES})+   # Parenthesized csv data (1 or more)
        \s*                            # Optional whitespace
        [δ=:\s]*                       # Optional NMR notation
        {NMR_PEAK_LIST}                # NMR peak list
    """,
    re.VERBOSE,
)

HRMS_PATTERN = re.compile(
    rf"HRMS(.{{1,100}})?found[\s:]+{POSITIVE_NUMBER}", re.DOTALL | re.IGNORECASE
)

CSV_NUMBERS = rf"{POSITIVE_NUMBER}(?:\s*,\s*{POSITIVE_NUMBER})+"
IR_PATTERN = re.compile(
    rf"(?:FT)?IR[^\d]{{1,20}}{CSV_NUMBERS}",
    re.IGNORECASE,
)

SEP_SPACE_PATTERN = re.compile(r"[,;\.][,;\.\s]+")


def remove_analysis_data(text: str) -> str:
    text = UNITS_PATTERN.sub("", text)
    text = NMR_PATTERN.sub("", text)
    text = HRMS_PATTERN.sub("", text)
    text = IR_PATTERN.sub("", text)
    text = SEP_SPACE_PATTERN.sub(" ", text)
    return text.strip()


def retrieve_excess_or_ratio(rotation_block: dict) -> str:
    article_id = rotation_block["article_id"]
    file_stem = rotation_block["file_stem"]
    data_block_path = DATA_BLOCKS_DIR / article_id / f"{file_stem}.json"
    with open(data_block_path, "r") as f:
        data_blocks = json.load(f)
    for data_block in data_blocks:
        delta = data_block["start"] - rotation_block["stop"]

        if (
            delta > 0  # the first data block after the rotation block
            and data_block["type"] == "rotation"  # is a rotation block
        ):
            return "unknown"

        if (
            delta > 0  # the first data block after the rotation block
            and data_block["type"] in ["excess", "ratio"]  # is an excess or ratio block
        ):
            if delta < SHORT_THRESHOLD_DISTANCE:
                return data_block["text"]

            if delta < LONG_THRESHOLD_DISTANCE:
                text_file = EXTRACTED_DIR / article_id / f"{file_stem}.txt"
                with open(text_file, "r") as f:
                    text = f.read()
                in_between = text[rotation_block["stop"] : data_block["start"]]
                in_between = remove_analysis_data(in_between).strip()
                if len(in_between) < SHORT_THRESHOLD_DISTANCE:
                    return data_block["text"]
                return "unknown"
            return "unknown"
    return "unknown"

In [ ]:
if COLLECTED_ROTATION_RESULTS_PATH.exists():
    raise FileExistsError(f"File {COLLECTED_ROTATION_RESULTS_PATH} already exists")

rotation_results = []
rotation_results_files = list(ROTATION_RESULTS_DIR.glob("*"))
for path in tqdm(rotation_results_files, desc="Processing articles"):
    if not (path.is_dir() and path.name.isdigit()):
        continue
    article_id = path.name
    for subdir in path.glob("*"):
        if not subdir.is_dir():
            continue
        stem = subdir.name
        for block_file in subdir.glob("*.json"):
            with open(block_file, "r") as f:
                block = json.load(f)
            if block["molecule"] == "unknown":
                continue
            extended_block = get_extended_block(
                block, block["molecule"], article_id, stem
            )
            if extended_block is None:
                continue
            if extended_block["excess_or_ratio"] == "unknown":
                extended_block["excess_or_ratio"] = retrieve_excess_or_ratio(
                    extended_block
                )
            rotation_results.append(extended_block)

rotation_results_df = pd.DataFrame(rotation_results)
rotation_results_df.to_csv(COLLECTED_ROTATION_RESULTS_PATH, index=False)
rotation_results_df

In [ ]:
rotation_results_df[rotation_results_df["excess_or_ratio"] != "unknown"]